In [1]:
import ray
import gym
import numpy as np
from os.path import join as pjoin

env_path = '/data/home/zhangjs/disk/project/verl-agent/agent_system/environments'
file_path = 'env_package/webshop/webshop/data/items_shuffle_1000.json'
attr_path = 'env_package/webshop/webshop/data/items_ins_v2_1000.json'

env_kwargs = {
    'observation_mode': 'text', 
    'num_products': None, 
    'human_goals': False,
    'file_path': '/data/home/zhangjs/disk/project/verl-agent/agent_system/environments/env_package/webshop/webshop/data/items_shuffle_1000.json',
    'attr_path': '/data/home/zhangjs/disk/project/verl-agent/agent_system/environments/env_package/webshop/webshop/data/items_ins_v2_1000.json'
} 


/diskpool/zhangjs/miniconda3/envs/verl-agent-webshop/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-10-05 13:25:26,918	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
Additionally, Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [2]:
class WebshopWorker:
    """Ray remote actor that replaces the worker function.
    Each actor hosts a *WebAgentTextEnv* instance.
    """
    
    def __init__(self, env_path, seed, env_kwargs):
        # Lazy import avoids CUDA initialisation issues
        import sys 
        import os 
        project_root = env_path
        sys.path.append(project_root)
        from web_agent_site.envs import WebAgentTextEnv # from web_agent_site.envs import WebAgentTextEnv  # noqa: WPS433 (runtime import)
        
        env_kwargs['seed'] = seed 
        self.env = gym.make('WebAgentTextEnv-v0', **env_kwargs) 
    
    def step(self, action):
        """Execute a step in the environment"""
        obs, reward, done, info = self.env.step(action)
        info = dict(info or {})  # make a *copy* so we can mutate safely
        info['available_actions'] = self.env.get_available_actions()
        info['task_score'] = reward
        
        # Redefine reward. We only use rule-based reward - win for 10, lose for 0.
        if done and reward == 1.0:
            info['won'] = True
            reward = 10.0
        else:
            info['won'] = False
            reward = 0

        return obs, reward, done, info
    
    def reset(self, idx):
        """Reset the environment with given session index"""
        obs, info = self.env.reset(session=idx) 
        info = dict(info or {}) 
        info['available_actions'] = self.env.get_available_actions()
        info['won'] = False
        return obs, info
    
    def render(self, mode_for_render):
        """Render the environment"""
        rendered = self.env.render(mode=mode_for_render)
        return rendered
    
    def get_available_actions(self):
        """Get available actions"""
        return self.env.get_available_actions()
    
    def get_goals(self):
        """Get environment goals"""
        return self.env.server.goals
    
    def close(self):
        """Close the environment"""
        self.env.close() 

In [3]:
# Basic Env Manager
class Env:
    def __init__(self, game_file):
        self.gamefile = game_file
        self.env_path = '/data/home/zhangjs/disk/project/verl-agent/agent_system/environments/env_package/webshop/webshop'
        self.env_kwargs = {
            'observation_mode': 'text', 
            'num_products': None, 
            'human_goals': False,
            'file_path': '/data/home/zhangjs/disk/project/verl-agent/agent_system/environments/env_package/webshop/webshop/data/items_shuffle_1000.json',
            'attr_path': '/data/home/zhangjs/disk/project/verl-agent/agent_system/environments/env_package/webshop/webshop/data/items_ins_v2_1000.json'
        } 
        env, obs, infos = self.build_env(
            gamefile=game_file,
            env_kwargs=self.env_kwargs,
        )
        self.env = env 
        self.start_obv = obs 
        self.start_infos = infos 
        self.last_command = []
        self.auto_reset = False
        self.is_done = False
    
    def step(self, action):
        obs, reward, done, info = self.env.step(action) 
        
        # 处理 info 可能为 None 的情况
        info = dict(info or {})
        available_actions = self.env.get_available_actions()  # 直接从环境获取

        # Formated the obs and actions
        formated_obs = self.format_obs(obs, add_task=False) 
        possible_actions = self.format_possible_actions(available_actions) 
        
        if done:
            self.is_done = True 

        self.last_command.append(
            {
                'action': action,
                'observation': formated_obs,
                'rewards': reward,
                'dones': done,
                'possible_commands': possible_actions,
                'game_file': self.gamefile
            }
        )
        
        copied_info = dict(info) 
        copied_info['admissible_commands'] = possible_actions
        copied_info['task_score'] = reward
        if done and reward == 1.0:
            copied_info['won'] = True
        else:
            copied_info['won'] = False

        return formated_obs, reward, done, copied_info
    
    def reset(self):
        obs, infos = self.env.reset(session=self.gamefile) 
        
        # 处理 infos 可能为 None 的情况
        infos = dict(infos or {})
        self.task = self.extract_task(obs) 
        formated_obs = self.format_obs(obs, add_task=True) 
        
        copied_infos = {}
        available_actions = self.env.get_available_actions()  # 直接从环境获取
        copied_infos['admissible_commands'] = self.format_possible_actions(available_actions)

        return formated_obs, copied_infos

    
    def build_env(self, gamefile, env_kwargs): # the gamefile is actually seed
        import sys 
        import os 
        sys.path.append(self.env_path) 

        from web_agent_site.envs import WebAgentTextEnv
        
        env_kwargs['seed'] = gamefile 
        env = gym.make('WebAgentTextEnv-v0', **env_kwargs) 
        
        obs, infos = env.reset(session=gamefile) 
        
        # 处理 infos 可能为 None 的情况
        infos = dict(infos or {})
        self.task = self.extract_task(obs) 
        formated_obs = self.format_obs(obs, add_task=True) 

        copied_infos = {}
        available_actions = env.get_available_actions()  # 直接从环境获取
        copied_infos['admissible_commands'] = self.format_possible_actions(available_actions)

        return env, formated_obs, copied_infos
    
    def extract_task(self, text_obs):
        parts = text_obs.split(" [SEP] ")
        task = parts[2]
        return task
    
    def format_obs(self, text_obs, add_task=True):
        parts = text_obs.split(" [SEP] ")
        # the index of self.tasks[i] in parts
        try:
            index = parts.index(self.task) 
            reformatted_obs = " [SEP] ".join(f"'{p}'" for p in parts[index+1:])
        except:
            reformatted_obs = text_obs
        if add_task:
            return reformatted_obs + '\n\nYour task is to: ' + self.task
        else:
            return reformatted_obs
    
    def format_possible_actions(self, possible_actions):
        actions = []
    
        for key in possible_actions.keys():
            if key not in ["has_search_bar", "clickables"]:
                raise ValueError(f"Unknown key in available actions: {key}")

        if possible_actions["has_search_bar"]:
            actions.append("search[<your query>]")

        for txt in possible_actions["clickables"]:
            actions.append(f"click[{txt}]")

        return actions
    
    def render(self, mode_for_render):
        """Render the environment"""
        rendered = self.env.render(mode=mode_for_render)
        return rendered
    
    def get_available_actions(self):
        """Get available actions"""
        return self.env.get_available_actions()
    
    def get_goals(self):
        """Get environment goals"""
        return self.env.server.goals
    
    def close(self):
        """Close the environment"""
        self.env.close()

In [4]:
env1 = Env(game_file=1)
env1.reset() 

/diskpool/zhangjs/miniconda3/envs/verl-agent-webshop/lib/python3.10/site-packages/faiss/loader.py:28: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  if LooseVersion(numpy.__version__) >= "1.19":
/diskpool/zhangjs/miniconda3/envs/verl-agent-webshop/lib/python3.10/site-packages/setuptools/_distutils/version.py:336: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  other = LooseVersion(other)
/diskpool/zhangjs/miniconda3/envs/verl-agent-webshop/lib/python3.10/site-packages/spacy/cli/_util.py:23: DeprecationWarning: Importing 'parser.split_arg_string' is deprecated, it will only be available in 'shell_completion' in Click 9.0.
  from click.parser import split_arg_string
/diskpool/zhangjs/miniconda3/envs/verl-agent-webshop/lib/python3.10/site-packages/weasel/util/config.py:8: DeprecationWarning: Importing 'parser.split_arg_string' is deprecated, it will only be available in 'shell_completion' in Cli

Products loaded.

Keys cleaned.

Attributes loaded.

100%|██████████| 1000/1000 [00:00<00:00, 100756.80it/s]

Loaded 6910 goals.



/diskpool/zhangjs/miniconda3/envs/verl-agent-webshop/lib/python3.10/site-packages/bs4/element.py:784: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  warnings.warn(
/diskpool/zhangjs/miniconda3/envs/verl-agent-webshop/lib/python3.10/site-packages/gym/envs/registration.py:619: UserWarning: WARN: Env check failed with the following message: You must specify an observation space (cf gym.spaces) cf https://github.com/openai/gym/blob/master/gym/spaces/
You can set `disable_env_checker=True` to disable this check.
  logger.warn(


("'Search'\n\nYour task is to: Find me men's loafers & slip-ons with rubber outsole, rubber sole with color: brown, and size: 9.5, and price lower than 70.00 dollars",
 {'admissible_commands': ['search[<your query>]', 'click[search]']})

In [ ]:
env_test = WebshopWorker(
    env_path='/data/home/zhangjs/disk/project/verl-agent/agent_system/environments/env_package/webshop/webshop',
    seed=0,
    env_kwargs=env_kwargs
)
obs,info = env_test.reset(1)
obs,info

In [ ]:
obs,info = env_test.reset(1)

In [ ]:
info 

In [ ]:
def extract_task(text_obs):
    parts = text_obs.split(" [SEP] ")
    task = parts[2]
    return task

task = extract_task(env_test.reset(999)[0])
task

In [ ]:
text_obs = env_test.reset(999)[0]
parts = text_obs.split(" [SEP] ")
# the index of self.tasks[i] in parts
try:
    index = parts.index(task) 
    reformatted_obs = " [SEP] ".join(f"'{p}'" for p in parts[index+1:])
except:
    reformatted_obs = text_obs 

reformatted_obs 

In [ ]:
env_test.get_available_actions()

In [ ]:
text_obs = env_test.step('search[wash]')[0] 
text_obs

In [ ]:
env_test.step('search[wash]') 

In [ ]:
def format_avail_actions(avail):
    actions = []
    
    for key in avail.keys():
        if key not in ["has_search_bar", "clickables"]:
            raise ValueError(f"Unknown key in available actions: {key}")

    if avail["has_search_bar"]:
        actions.append("search[<your query>]")

    for txt in avail["clickables"]:
        actions.append(f"click[{txt}]")

    return actions

format_avail_actions(env_test.step('search[wash]')[-1]['available_actions'])

In [ ]:
parts = text_obs.split(" [SEP] ")
# the index of self.tasks[i] in parts
try:
    index = parts.index(task) 
    reformatted_obs = " [SEP] ".join(f"'{p}'" for p in parts[index+1:])
except:
    reformatted_obs = text_obs 

reformatted_obs 

In [ ]:
def extract_task(self, text_obs: List[str]):
    tasks = []
    for obs in text_obs:
        parts = obs.split(" [SEP] ")
        assert parts[1]=='Instruction:'
        tasks.append(parts[2])
    return tasks

env_test.step('search[wash]')[0]

def format_obs(text_obs): 
    postprocess_text_obs = [] 
    for i in range(len(text_obs)):
        parts = text_obs[i].split(" [SEP] ")
        # the index of self.tasks[i] in parts
        try:
            index = parts.index(self.tasks[i])
            reformatted_obs = " [SEP] ".join(f"'{p}'" for p in parts[index+1:])
        except:
            reformatted_obs = text_obs[i]

        postprocess_text_obs.append(reformatted_obs)

    return postprocess_text_obs

In [ ]:
WEBSHOP_TEMPLATE_NO_HIS = """
You are an expert autonomous agent operating in the WebShop e‑commerce environment. 
Your task is to: {task_description}.
Your current observation is: {current_observation}.
Your admissible actions of the current situation are: 
[
{available_actions}
].

Now it's your turn to take one action for the current step.
You should first reason step-by-step about the current situation, then think carefully which admissible action best advances the shopping goal. This reasoning process MUST be enclosed within <think> </think> tags. 
Once you've finished your reasoning, you should choose an admissible action for current step and present it within <action> </action> tags.
"""

WEBSHOP_TEMPLATE = """
You are an expert autonomous agent operating in the WebShop e‑commerce environment.
Your task is to: {task_description}.
Prior to this step, you have already taken {step_count} step(s). Below are the most recent {history_length} observations and the corresponding actions you took: {action_history}
You are now at step {current_step} and your current observation is: {current_observation}.
Your admissible actions of the current situation are: 
[
{available_actions}
].

Now it's your turn to take one action for the current step.
You should first reason step-by-step about the current situation, then think carefully which admissible action best advances the shopping goal. This reasoning process MUST be enclosed within <think> </think> tags. 
Once you've finished your reasoning, you should choose an admissible action for current step and present it within <action> </action> tags.
"""

In [ ]:
env_test.reset(999)[1]

In [ ]:
env_test.get_available_actions()

In [ ]:
env_test.reset(1145) 